# DSA-02 · Binary search on the answer (LC 875, LC 1011)

[Open in Colab](https://colab.research.google.com/github/BuiltBySwap/engineering-lab/blob/main/docs/topics/DSA-02.ipynb)

**Code:** [kotlin-dsa-kata](https://github.com/BuiltBySwap/kotlin-dsa-kata)

**Status card** (edit it; the Progress and Tags pages are built from it):

```yaml
id: DSA-02
title: "Binary search on the answer (LC 875, LC 1011)"
track: dsa
date: 2026-10-12
code: "kotlin-dsa-kata"
status: partial  # not-started | in-progress | partial | done
score: ""
insight: "Binary search works on any question where 'does value X work?' flips only once as X grows (no, no, yes, yes): keep the middle when it works (hi = mid), drop it when it fails (lo = mid + 1); only the check and the bounds change between problems."
tags: [DSA-02, dsa, binary-search, leetcode, kotlin, monotonic]
pending:
  - "MQ6 (how does Zygote start?) in my own words"
  - "English (scalable, robust) and the role lens"
  - "Block A: LC 410, LC 1482, LC 162 (the first two use this same pattern)"
```

## Goal
Binary search without a sorted array: search the **range of possible answers**, using a yes/no check that flips only once.

## LC 875 · Koko eating bananas
Piles `[3, 6, 7, 11]`, `h = 8` hours. Each hour Koko eats up to `k` bananas from **one** pile (a smaller pile is finished and she waits out the hour). What is the **smallest speed `k`** that finishes in 8 hours?

**The check:** hours needed at speed `k` = sum of `ceil(pile ÷ k)`. In integers, `ceil(pile ÷ k)` is `(pile + k - 1) / k`.

| Speed k | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 |
|---|---|---|---|---|---|---|---|---|---|---|---|
| Hours | 27 | 15 | 10 | 8 | 8 | 6 | 5 | 5 | 5 | 5 | 4 |
| Works (≤ 8)? | no | no | no | **yes** | yes | yes | yes | yes | yes | yes | yes |

A faster eater never needs more hours, so the last row is **no, no, no, yes, yes, yes**: it flips once. The answer is the **first yes**. That property is called *monotonic*.

**The search** (range 1 to the biggest pile, 11, because that speed certainly works):

| Range | Middle | Hours | Verdict | Keep |
|---|---|---|---|---|
| 1 to 11 | 6 | 6 | works | 1 to 6 |
| 1 to 6 | 3 | 10 | too slow | 4 to 6 |
| 4 to 6 | 5 | 8 | works | 4 to 5 |
| 4 to 5 | 4 | 8 | works | 4 to 4 |

Four tries instead of eleven. **Why `hi = mid`:** when the middle works it might be the answer, so keep it. When it fails it is not the answer, so `lo = mid + 1`.

## LC 1011 · Capacity to ship packages within D days
Packages ship **in order**; each day the ship carries a run of packages up to its **capacity**. Smallest capacity that ships `[1..10]` in 5 days? Capacity 15 works (`[1..5]`, `[6,7]`, `[8]`, `[9]`, `[10]`) and 14 needs 6 days, so the answer is **15**.

**The bounds come from the problem:** `lo` = the heaviest package (a smaller ship can never carry it), `hi` = the total weight (one day always works).

| | LC 875 (Koko) | LC 1011 (ship) |
|---|---|---|
| What we search | the speed | the capacity |
| The check | hours at this speed ≤ h | days at this capacity ≤ D |
| `lo` | 1 | the heaviest package |
| `hi` | the biggest pile | the total weight |
| The `while` loop | identical | identical |

**Recognise it:** "the smallest value such that X works", "minimise the maximum" (the ship's biggest daily load), "maximise the minimum".

**Cost:** each try reads every item once (n steps); the number of tries is about log₂ of the range (about 30 for a billion). The total is about n × 30, not n × 1,000,000,000.

In [ ]:
# Setup: run once per session. Adds the %%kotlin cell magic. Works in Colab and in VS Code / Jupyter.
import os, urllib.request
SRC = next((p for p in ("tools/kotlin_magic.py", "../../tools/kotlin_magic.py") if os.path.exists(p)), None)
if SRC is None:                                   # e.g. Colab: download it from GitHub
    SRC = "kotlin_magic.py"
    urllib.request.urlretrieve("https://raw.githubusercontent.com/BuiltBySwap/engineering-lab/main/tools/kotlin_magic.py", SRC)
%run $SRC

In [ ]:
%%kotlin
import kotlin.test.assertEquals

fun hoursNeeded(piles: IntArray, speed: Int): Long {
    var hours = 0L                               // Long: the total can pass Int's limit of about 2.1 billion
    for (pile in piles) hours += (pile + speed - 1) / speed      // round up
    return hours
}

fun minEatingSpeed(piles: IntArray, h: Int): Int {
    var lo = 1
    var hi = piles.max()                         // this speed certainly works
    while (lo < hi) {
        val mid = lo + (hi - lo) / 2
        if (hoursNeeded(piles, mid) <= h) hi = mid else lo = mid + 1
    }
    return lo
}

fun daysNeeded(weights: IntArray, capacity: Int): Int {
    var days = 1
    var load = 0
    for (w in weights) {
        if (load + w > capacity) { days++; load = 0 }
        load += w
    }
    return days
}

fun shipWithinDays(weights: IntArray, days: Int): Int {
    var lo = weights.max()                       // must carry the heaviest package
    var hi = weights.sum()                       // everything in one day always works
    while (lo < hi) {
        val mid = lo + (hi - lo) / 2
        if (daysNeeded(weights, mid) <= days) hi = mid else lo = mid + 1
    }
    return lo
}

assertEquals(4, minEatingSpeed(intArrayOf(3, 6, 7, 11), 8))
assertEquals(30, minEatingSpeed(intArrayOf(30, 11, 23, 4, 20), 5))
assertEquals(23, minEatingSpeed(intArrayOf(30, 11, 23, 4, 20), 6))
assertEquals(10_000_000_000_000L, hoursNeeded(IntArray(10_000) { 1_000_000_000 }, 1))      // needs Long
assertEquals(15, shipWithinDays(intArrayOf(1, 2, 3, 4, 5, 6, 7, 8, 9, 10), 5))
assertEquals(6, shipWithinDays(intArrayOf(3, 2, 2, 4, 1, 4), 3))
assertEquals(3, shipWithinDays(intArrayOf(1, 2, 3, 1, 1), 4))
println("LC 875 and LC 1011: all checks passed")

## Break it on purpose (the experiment that makes `hi = mid` stick)
Change `hi = mid` to `hi = mid - 1` in `minEatingSpeed`: the search throws away a middle that works, and it might have been the answer. Change `lo = weights.max()` to `lo = 1` in `shipWithinDays`: the check is wrong for capacities below the heaviest package. The cell below runs both mistakes against a brute force and lists the tests that catch them (the same cases as the tests in the kata repo).

In [ ]:
def hours(p, k): return sum((x + k - 1) // k for x in p)
def days(w, c):
    d, load = 1, 0
    for x in w:
        if load + x > c: d += 1; load = 0
        load += x
    return d

def koko(p, h, keep_mid=True):
    lo, hi = 1, max(p)
    while lo < hi:
        mid = (lo + hi) // 2
        if hours(p, mid) <= h: hi = mid if keep_mid else mid - 1       # the mistake: mid - 1
        else: lo = mid + 1
    return lo

def ship(w, d, lo_is_max=True):
    lo, hi = (max(w) if lo_is_max else 1), sum(w)                        # the mistake: lo = 1
    while lo < hi:
        mid = (lo + hi) // 2
        if days(w, mid) <= d: hi = mid
        else: lo = mid + 1
    return lo

koko_cases = {"example1": ([3,6,7,11], 8, 4), "example2": ([30,11,23,4,20], 5, 30), "example3": ([30,11,23,4,20], 6, 23),
              "singlePileInOneHour": ([5], 1, 5), "singlePileInTwoHours": ([5], 2, 3), "pileOfOne": ([1], 1, 1)}
ship_cases = {"example1": (list(range(1, 11)), 5, 15), "example2": ([3,2,2,4,1,4], 3, 6), "example3": ([1,2,3,1,1], 4, 3),
              "singlePackage": ([7], 1, 7), "eachPackageOwnDay": ([1,2,3,4,5], 5, 5), "oneDay": ([1,2,3,4,5], 1, 15)}

for name, (p, h, want) in koko_cases.items(): assert koko(p, h) == want, name          # the real solution passes everything
for name, (w, d, want) in ship_cases.items(): assert ship(w, d) == want, name

broken_koko = [n for n, (p, h, want) in koko_cases.items() if koko(p, h, keep_mid=False) != want]
broken_ship = [n for n, (w, d, want) in ship_cases.items() if ship(w, d, lo_is_max=False) != want]
print("hi = mid - 1 breaks:", broken_koko)
print("lo = 1 breaks      :", broken_ship)
assert broken_koko == ["example3", "singlePileInTwoHours"]
assert broken_ship == ["example3", "eachPackageOwnDay"]

## MQ6 · How does Zygote start?
`init` reads `init.zygote64.rc` and starts the `zygote` service, which runs `app_process64` (with the flags `--zygote --start-system-server`). `app_process` starts the ART runtime and calls `ZygoteInit.main`: it **preloads** classes and resources once, opens the **Zygote socket**, **forks `system_server`**, and then waits on the socket to fork an app process on request. (Answer from the tutor: write your own version.)

## Quiz and teaching note
The first, abstract explanation ("search the answer space", "monotonic predicate") did not land (two "not sure" answers). The **table of hours for every speed** did: seeing the no, no, no, yes, yes row made the idea obvious. Lesson for next time: start with numbers, name the idea after.

## English and thinking
Words: **scalable** (grows without breaking), **robust** (strong under stress). Grammar: past simple vs present perfect.
✏️ Three sentences about today with both words, one past simple and one present perfect.

## Role lens · Mobile Dev
✏️ Where would "binary search on the answer" show up in a real app or backend? (Example shape: *a rate limit, a retry delay, a batch size: the smallest value that keeps latency under a limit.*)

## Next
**Block A (optional):** LC 410 and LC 1482 use exactly this pattern; LC 162 is a binary search on the slope. Practice track continues Mondays in `kotlin-dsa-kata`.